In [1]:
import torch
import numpy as np
import pandas as pd
import os
from google.colab import userdata, drive

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/PRISM'
EMBED_DIR = f'{BASE_DIR}/embeddings/midnight/pcam'
RESULTS_DIR = f'{BASE_DIR}/results'
os.makedirs(EMBED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.1 GB


In [2]:
!pip install transformers timm einops huggingface_hub scikit-learn -q

from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("Login OK")

Login OK


In [4]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained(
    "kaiko-ai/midnight",
    trust_remote_code=True
)
model = model.cuda()
model.eval()
print("MIDNIGHT loaded!")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

model.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

MIDNIGHT loaded!
GPU memory: 4.55 GB


In [5]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
])
print("MIDNIGHT transform ready")

MIDNIGHT transform ready


In [6]:
from torchvision.datasets import PCAM
from torch.utils.data import DataLoader

PCAM_DIR = '/content/drive/MyDrive/PRISM/datasets/pcam'

train_dataset = PCAM(root=PCAM_DIR, split='train', download=False, transform=transform)
val_dataset   = PCAM(root=PCAM_DIR, split='val',   download=False, transform=transform)
test_dataset  = PCAM(root=PCAM_DIR, split='test',  download=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")
print(f"Test:  {len(test_dataset):,}")

Train: 262,144
Val:   32,768
Test:  32,768


In [8]:
from tqdm import tqdm

def extract_features(model, loader, device='cuda'):
    all_features = []
    all_labels = []
    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting"):
            images = images.to(device)
            output = model(images)
            # MIDNIGHT pooler_output kullan
            features = output.pooler_output
            features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_features), np.concatenate(all_labels)

print("Extracting train features...")
train_features, train_labels = extract_features(model, train_loader)
print(f"Train: {train_features.shape}")

print("Extracting val features...")
val_features, val_labels = extract_features(model, val_loader)
print(f"Val: {val_features.shape}")

print("Extracting test features...")
test_features, test_labels = extract_features(model, test_loader)
print(f"Test: {test_features.shape}")

Extracting train features...


Extracting: 100%|██████████| 1024/1024 [3:01:04<00:00, 10.61s/it]


Train: (262144, 1536)
Extracting val features...


Extracting: 100%|██████████| 128/128 [22:34<00:00, 10.58s/it]


Val: (32768, 1536)
Extracting test features...


Extracting: 100%|██████████| 128/128 [19:46<00:00,  9.27s/it]

Test: (32768, 1536)


In [9]:
np.save(f'{EMBED_DIR}/train_features.npy', train_features)
np.save(f'{EMBED_DIR}/train_labels.npy', train_labels)
np.save(f'{EMBED_DIR}/val_features.npy', val_features)
np.save(f'{EMBED_DIR}/val_labels.npy', val_labels)
np.save(f'{EMBED_DIR}/test_features.npy', test_features)
np.save(f'{EMBED_DIR}/test_labels.npy', test_labels)
print(f"Saved to: {EMBED_DIR}")

Saved to: /content/drive/MyDrive/PRISM/embeddings/midnight/pcam


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss

def compute_ece(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() > 0:
            ece += mask.sum() * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece / len(y_true)

def run_label_fraction(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_l) * fraction)
    idx = np.random.choice(len(train_l), n, replace=False)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    y_prob = clf.predict_proba(test_f)[:, 1]
    y_pred = clf.predict(test_f)
    return {
        'model': 'MIDNIGHT', 'dataset': 'PCam',
        'fraction': fraction, 'n_samples': n, 'seed': seed,
        'auroc': roc_auc_score(test_l, y_prob),
        'f1':    f1_score(test_l, y_pred),
        'brier': brier_score_loss(test_l, y_prob),
        'ece':   compute_ece(test_l, y_prob),
    }

fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
seeds     = [42, 123, 456]
results   = []

for frac in fractions:
    for seed in seeds:
        res = run_label_fraction(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        results.append(res)
        print(f"  {frac*100:.0f}% | seed {seed} | AUROC: {res['auroc']:.4f} | ECE: {res['ece']:.4f} | Brier: {res['brier']:.4f}")

df = pd.DataFrame(results)
print("\n--- Summary (mean over seeds) ---")
print(df.groupby('fraction')[['auroc','ece','brier','f1']].mean().round(4))

  1% | seed 42 | AUROC: 0.9520 | ECE: 0.0544 | Brier: 0.0872
  1% | seed 123 | AUROC: 0.9428 | ECE: 0.0561 | Brier: 0.0919
  1% | seed 456 | AUROC: 0.9400 | ECE: 0.0562 | Brier: 0.0957
  5% | seed 42 | AUROC: 0.9570 | ECE: 0.0478 | Brier: 0.0776
  5% | seed 123 | AUROC: 0.9540 | ECE: 0.0536 | Brier: 0.0804
  5% | seed 456 | AUROC: 0.9523 | ECE: 0.0520 | Brier: 0.0817
  10% | seed 42 | AUROC: 0.9551 | ECE: 0.0494 | Brier: 0.0768
  10% | seed 123 | AUROC: 0.9557 | ECE: 0.0501 | Brier: 0.0766
  10% | seed 456 | AUROC: 0.9565 | ECE: 0.0505 | Brier: 0.0763
  25% | seed 42 | AUROC: 0.9539 | ECE: 0.0510 | Brier: 0.0758
  25% | seed 123 | AUROC: 0.9566 | ECE: 0.0533 | Brier: 0.0748
  25% | seed 456 | AUROC: 0.9553 | ECE: 0.0531 | Brier: 0.0760
  50% | seed 42 | AUROC: 0.9551 | ECE: 0.0539 | Brier: 0.0747
  50% | seed 123 | AUROC: 0.9532 | ECE: 0.0531 | Brier: 0.0756
  50% | seed 456 | AUROC: 0.9550 | ECE: 0.0544 | Brier: 0.0738
  100% | seed 42 | AUROC: 0.9534 | ECE: 0.0558 | Brier: 0.0740
  1

In [11]:
from scipy.optimize import minimize_scalar

def find_temperature(val_logits, val_labels):
    def nll(T):
        scaled = val_logits / T
        exp_s  = np.exp(scaled - scaled.max(axis=1, keepdims=True))
        probs  = exp_s / exp_s.sum(axis=1, keepdims=True)
        return -np.log(probs[np.arange(len(val_labels)), val_labels] + 1e-10).mean()
    return minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded').x

def run_temperature_scaling(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_l) * fraction)
    idx = np.random.choice(len(train_l), n, replace=False)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    val_logits = clf.decision_function(val_f).reshape(-1,1)
    val_logits = np.hstack([-val_logits, val_logits])
    T          = find_temperature(val_logits, val_l)
    test_prob_raw = clf.predict_proba(test_f)[:, 1]
    ece_raw       = compute_ece(test_l, test_prob_raw)
    test_logits = clf.decision_function(test_f).reshape(-1,1)
    test_logits = np.hstack([-test_logits, test_logits])
    scaled      = test_logits / T
    exp_s       = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    test_prob_s = (exp_s / exp_s.sum(axis=1, keepdims=True))[:, 1]
    ece_scaled  = compute_ece(test_l, test_prob_s)
    return {
        'model': 'MIDNIGHT', 'dataset': 'PCam',
        'fraction': fraction, 'seed': seed,
        'temperature': T,
        'ece_raw': ece_raw, 'ece_scaled': ece_scaled,
        'ece_improvement': ece_raw - ece_scaled,
        'auroc': roc_auc_score(test_l, test_prob_raw),
        'brier_raw': brier_score_loss(test_l, test_prob_raw),
        'brier_scaled': brier_score_loss(test_l, test_prob_s),
    }

ts_results = []
for frac in fractions:
    for seed in seeds:
        res = run_temperature_scaling(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        ts_results.append(res)

df_ts = pd.DataFrame(ts_results)
print("--- Temperature Scaling Results ---")
print(df_ts.groupby('fraction')[['temperature','ece_raw','ece_scaled','ece_improvement']].mean().round(4))

--- Temperature Scaling Results ---
          temperature  ece_raw  ece_scaled  ece_improvement
fraction                                                   
0.01           1.5425   0.0556      0.0573          -0.0017
0.05           1.8582   0.0511      0.0539          -0.0027
0.10           2.0151   0.0500      0.0497           0.0003
0.25           2.1797   0.0525      0.0496           0.0029
0.50           2.3528   0.0538      0.0487           0.0051
1.00           2.3799   0.0558      0.0509           0.0049


In [12]:
df.to_csv(f'{RESULTS_DIR}/midnight_pcam_results.csv', index=False)
df_ts.to_csv(f'{RESULTS_DIR}/midnight_pcam_temperature_scaling.csv', index=False)
print("Saved!")

Saved!
